In [ ]:
from models.backbones import Backbone
from models.boq import BoQ
import torch

import warnings
warnings.filterwarnings("ignore")

backbone = Backbone(
        backbone_name="dinov2_vits14",
        # backbone_name="resnet18",
        pretrained=True,
        unfreeze_n_blocks= 1,
        crop_last_block=True)

aggregator = BoQ(in_channels=backbone.out_channels, 
                 proj_channels=256, 
                 num_queries=64, 
                 num_layers=2,
                 row_dim=4096,
                 linear_dims=[1024, 1024, 512]
                 )

device = "cuda" if torch.cuda.is_available() else "cpu"

Using cache found in C:\Users\nithish.kumar/.cache\torch\hub\facebookresearch_dinov2_main


In [5]:
import lightning as L

class BoQModel(L.LightningModule):
    def __init__(self, backbone, aggregator):
        super().__init__()
        self.backbone = backbone 
        self.aggregator = aggregator

    
    def forward(self, x): 
        x = self.backbone(x)
        x, _ = self.aggregator(x)
        return x

In [8]:
checkpoint_path = r"D:\shelf_audit\models\BOQ\DINO\boq-epoch=259-avg_train_loss_epoch=0.0002.ckpt"

model = BoQModel.load_from_checkpoint(checkpoint_path, backbone=backbone, aggregator=aggregator)
model.eval().to(device)


BoQModel(
  (backbone): Backbone(
    (net): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (blocks): ModuleList(
        (0-11): 12 x NestedTensorBlock(
          (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
          (attn): MemEffAttention(
            (qkv): Linear(in_features=384, out_features=1152, bias=True)
            (proj): Linear(in_features=384, out_features=384, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=384, out_features=1536, bias=True)
            (act): GELU(approximate='none')
            (fc2): Linear(in_features=1536, out_features=384, bias=True)
            (drop): Dropout(p=0.0, inplace=False)
     